# Notebook 06 — Model Training and Local Resolution

**Responsibilities (plan §12.7):**
- Feature preparation and pruning → frozen `core_frozen` subset
- Split loading and leakage verification
- Baseline models (GBDT, RandomForest, LogisticRegression)
- Optional: LightGBM / XGBoost / CatBoost model family comparison
- Probability calibration (sigmoid vs isotonic, chosen by Brier score on cal split)
- **Threshold selection on calibration split only** (never touches test)
- Multiple threshold policies: max_f1, max_fbeta, precision_floor, recall_floor
- Optional: Optuna hyperparameter search with GroupKFold spatial CV
- Evaluation on test split with full metric suite + stratified breakdowns
- SHAP TreeSHAP + beeswarm + dependence + waterfall plots
- Permutation importance (model-agnostic)
- Threshold sweep report
- Error taxonomy (FP/FN by isolation, close-pair, cluster, SNR bin)
- `model_predictions.parquet` (§3.11)
- `feature_stats.parquet` enriched (§3.12)
- Markdown model card
- Visual QA overlays: crops with annotation + model predictions, all channels

## Toggle flags (set before running)
- `CFG['run_lightgbm_family']` — LightGBM/XGBoost/CatBoost comparison  
- `CFG['run_optuna']` — Optuna hyperparameter search  
- `CFG['run_shap']` — SHAP analysis  
- `CFG['run_permutation_importance']` — permutation importance  
- `CFG['run_visual_qa']` — crop overlay visualization

**Currently: all optional stages toggled OFF** — enable when you have more annotations.

## Key design decisions
- **Threshold chosen on calibration split, reported on test** — no optimistic bias
- **GroupKFold CV** respects spatial blocks — no leakage across crops  
- **Calibration**: both sigmoid and isotonic fitted; winner chosen by Brier score  
  Sigmoid preferred on near-tie (can't overfit small cal set)  
- **Crops with zero annotations** are included as all-negative supervision rows  
- **ECE reported with caveat**: unreliable under heavy class imbalance; use Brier + reliability diagram instead

In [ ]:
# -------------------------------------------------------------------
# REPO DISCOVERY
# -------------------------------------------------------------------
from pathlib import Path

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / ".git").exists() or (p / "registries").exists():
            return p
    raise RuntimeError("Could not locate repo root.")

REPO_ROOT = find_repo_root()
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
# -------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------
CFG = {
    # ── Directory layout ───────────────────────────────────────────────────
    "tables_dir":     "tables",
    "artifact_dir":   "artifacts/reports/nb06_model",
    "model_card_dir": "artifacts/model_cards",
    "manifest_dir":   "artifacts/manifests",
    "models_dir":     "artifacts/models",

    # ── Input overrides (None = autodiscover latest from tables/) ──────────
    "supervision_table_path_override": None,
    "splits_path_override":            None,
    "feature_stats_path_override":     None,
    "candidate_union_path_override":   None,

    # ── Toggle flags ───────────────────────────────────────────────────────
    # Set to True when you have enough annotations (50+ positives per split)
    "run_lightgbm_family":       False,   # LightGBM / XGBoost / CatBoost comparison
    "run_optuna":                False,   # Bayesian hyperparameter search
    "run_shap":                  True,    # SHAP analysis (fast even with few samples)
    "run_permutation_importance": True,   # Permutation importance
    "run_visual_qa":             True,    # Crop overlay visualization

    # ── Feature pruning ────────────────────────────────────────────────────
    "min_non_null_frac":          0.50,   # drop features with more NaN than this on train
    "near_constant_freq_threshold": 0.9995,  # drop if top value frequency > this
    "correlation_threshold":      0.95,   # Pearson |r| threshold for corr filter
    "clip_quantile":              0.999,  # clip features to this quantile range (train)
    "min_univariate_auc":         0.50,   # drop features with reflected AUC < this

    # ── Baseline model hyperparameters (frozen) ───────────────────────────
    "gbdt_baseline_params": {
        "n_estimators": 500, "max_depth": 4, "learning_rate": 0.05,
        "subsample": 0.8, "max_features": "sqrt", "min_samples_leaf": 5,
        "random_state": 42,
    },
    "rf_baseline_params": {
        "n_estimators": 500, "max_depth": None, "min_samples_leaf": 5,
        "max_features": "sqrt", "random_state": 42, "n_jobs": -1,
    },
    "lr_baseline_params": {
        "C": 1.0, "max_iter": 2000, "random_state": 42, "solver": "lbfgs",
    },

    # ── Threshold selection ────────────────────────────────────────────────
    # ALWAYS selected on calibration split, NEVER on test
    # Options: "max_f1" | "max_fbeta" | "precision_floor" | "recall_floor" | "youden"
    "threshold_policy":          "max_f1",
    "threshold_fbeta_beta":       1.0,     # beta for max_fbeta (>1 = recall-weighted)
    "threshold_precision_floor":  0.80,    # used with precision_floor policy
    "threshold_recall_floor":     0.80,    # used with recall_floor policy

    # ── Calibration ────────────────────────────────────────────────────────
    "calibration_tol":            0.002,   # prefer sigmoid unless isotonic wins by > this

    # ── Optuna (used only if run_optuna=True) ─────────────────────────────
    "optuna_n_trials":            20,
    "optuna_n_cv_folds":          4,       # GroupKFold splits within training
    "optuna_max_trees_search":    150,     # n_estimators during search (refit with 500)
    "optuna_timeout_sec":         None,
    "optuna_min_train_crops":     4,       # skip if fewer train crops than this

    # ── LightGBM family (used only if run_lightgbm_family=True) ──────────
    "lgb_family_models":          ["lightgbm"],  # add "xgboost", "catboost" if installed
    "lgb_optuna_trials":          30,

    # ── Evaluation ─────────────────────────────────────────────────────────
    "threshold_sweep_steps":      99,
    "isolation_radius_px":        15.0,

    # ── SHAP ───────────────────────────────────────────────────────────────
    "shap_max_rows":              500,

    # ── Permutation importance ─────────────────────────────────────────────
    "permutation_n_repeats":      10,

    # ── Visual QA ──────────────────────────────────────────────────────────
    "qa_n_crops":                 10,
    "qa_circle_radius_px":        12,
    "qa_display_lo_pct":          1,
    "qa_display_hi_pct":          99,

    # ── Persistence ────────────────────────────────────────────────────────
    "write_model_predictions":    True,
    "write_feature_stats":        True,
    "write_models":               True,
    "write_reports":              True,
    "write_model_card":           True,

    # ── Provenance ─────────────────────────────────────────────────────────
    "model_registry_version":     "v1_nb06_baseline",
    "random_seed":                42,
}
CFG

In [ ]:
# -------------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------------
import gc, hashlib, json, math, os, time, warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import tifffile
import re
import yaml
from scipy.spatial import cKDTree

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance as sklearn_perm_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, log_loss, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Optional packages — warn but don't crash
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    warnings.warn("shap not installed — SHAP skipped. pip install shap")

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False
    warnings.warn("optuna not installed — tuning skipped. pip install optuna")

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    import catboost as cb
    CB_AVAILABLE = True
except ImportError:
    CB_AVAILABLE = False

print(f"shap={SHAP_AVAILABLE}  optuna={OPTUNA_AVAILABLE}  lgb={LGB_AVAILABLE}  xgb={XGB_AVAILABLE}  catboost={CB_AVAILABLE}")

In [ ]:
# -------------------------------------------------------------------
# HELPERS
# -------------------------------------------------------------------
def ts_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

def log(msg: str) -> None:
    print(f"[{ts_utc()}] {msg}", flush=True)

def ensure_dir(path) -> Path:
    p = Path(path); p.mkdir(parents=True, exist_ok=True); return p

def make_run_id(prefix="nb06") -> str:
    t = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    h = hashlib.sha1(f"{t}|{os.getpid()}|nb06".encode()).hexdigest()[:8]
    return f"{prefix}_{t}_{h}"

def sha1_text(x: str) -> str:
    return hashlib.sha1(x.encode()).hexdigest()

def safe_to_parquet(df: pd.DataFrame, path) -> Path:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
    except Exception as e:
        csv = path.with_suffix(".csv")
        log(f"[warn] parquet failed ({e}); writing CSV -> {csv.name}")
        df.to_csv(csv, index=False); return csv
    return path

def write_json(obj, path) -> Path:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    return path

def latest_matching_file(directory: Path, patterns: list):
    candidates = []
    if not directory.exists(): return None
    for pat in patterns:
        candidates.extend(directory.glob(pat))
    candidates = [p for p in candidates if p.is_file()]
    if not candidates: return None
    candidates.sort(key=lambda p: (p.stat().st_mtime, str(p)), reverse=True)
    return candidates[0]

def ece_binary(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0: continue
        ece += (mask.sum() / len(y_true)) * abs(float(y_true[mask].mean()) - float(y_prob[mask].mean()))
    return float(ece)

def eval_metrics(y_true: np.ndarray, y_prob: np.ndarray,
                 threshold: float = 0.5, sample_weight=None) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    out = {}
    try: out["roc_auc"]       = float(roc_auc_score(y_true, y_prob, sample_weight=sample_weight))
    except: out["roc_auc"]    = np.nan
    try: out["avg_precision"] = float(average_precision_score(y_true, y_prob, sample_weight=sample_weight))
    except: out["avg_precision"] = np.nan
    try: out["brier"]         = float(brier_score_loss(y_true, y_prob, sample_weight=sample_weight))
    except: out["brier"]      = np.nan
    try: out["logloss"]       = float(log_loss(y_true, np.clip(y_prob, 1e-8, 1-1e-8), sample_weight=sample_weight))
    except: out["logloss"]    = np.nan
    out["ece"]       = float(ece_binary(y_true, y_prob))
    out["f1"]        = float(f1_score(y_true, y_pred, zero_division=0))
    out["precision"] = float(precision_score(y_true, y_pred, zero_division=0))
    out["recall"]    = float(recall_score(y_true, y_pred, zero_division=0))
    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        out["tn"], out["fp"], out["fn"], out["tp"] = int(tn), int(fp), int(fn), int(tp)
    except:
        out["tn"] = out["fp"] = out["fn"] = out["tp"] = 0
    out["threshold"] = float(threshold)
    out["n_pos"]     = int(y_true.sum())
    out["n_total"]   = int(len(y_true))
    out["pos_rate"]  = float(y_true.mean())
    return out

def choose_threshold(y_true: np.ndarray, y_prob: np.ndarray,
                     policy: str = "max_f1",
                     fbeta: float = 1.0,
                     precision_floor: float = 0.80,
                     recall_floor: float = 0.80) -> dict:
    """
    Select threshold on calibration split ONLY.
    Never call this on the test split.
    """
    thresholds = np.unique(np.clip(y_prob, 0, 1))
    thresholds = np.concatenate([[0.0], thresholds, [1.0]])
    best, best_obj = None, -np.inf
    for thr in thresholds:
        m = eval_metrics(y_true, y_prob, threshold=thr)
        if policy == "max_f1":
            obj = m["f1"]
        elif policy == "max_fbeta":
            pr, rc = m["precision"], m["recall"]
            obj = (1 + fbeta**2) * pr * rc / max(fbeta**2 * pr + rc, 1e-12)
        elif policy == "precision_floor":
            obj = m["recall"] if m["precision"] >= precision_floor else -np.inf
        elif policy == "recall_floor":
            obj = m["precision"] if m["recall"] >= recall_floor else -np.inf
        else:  # youden
            obj = m["recall"] + m["precision"] - 1.0
        if obj > best_obj:
            best_obj, best = obj, {"threshold": float(thr), **m, "objective": float(obj)}
    return best

def _infer_group(col: str) -> str:
    if col.startswith(("photo_", "bpanel_")): return "patch_photometry"
    if col.startswith(("resp_", "psf_")):     return "detector_response"
    if any(col.startswith(p) for p in ["n_neighbors", "nn", "local_density", "border_",
        "cluster_", "nearest_", "nms_", "crop_n_", "crop_candidate"]):
        return "spatial_geometric"
    if any(col.startswith(p) for p in ["vote_", "proposer_", "family_", "union_",
        "consensus_", "top_score_"]): return "consensus"
    if col.startswith("barcode_"): return "barcode_multichannel"
    if col.startswith("crop_"):    return "crop_regime"
    if col.startswith("shape_"):   return "symmetry_shape"
    if any(col.startswith(p) for p in ["rank_", "mad_std_", "ratio_", "support_times"]):
        return "interaction_derived"
    return "other"

RUN_ID       = make_run_id("nb06")
TABLES_DIR   = ensure_dir(REPO_ROOT / CFG["tables_dir"])
REPORT_DIR   = ensure_dir(REPO_ROOT / CFG["artifact_dir"])
MODEL_DIR    = ensure_dir(REPO_ROOT / CFG["models_dir"])
CARD_DIR     = ensure_dir(REPO_ROOT / CFG["model_card_dir"])
MANIFEST_DIR = ensure_dir(REPO_ROOT / CFG["manifest_dir"])
log(f"RUN_ID = {RUN_ID}")

In [ ]:
# -------------------------------------------------------------------
# DISCOVERY AND LOAD
# -------------------------------------------------------------------
log("Discovering inputs ...")

def _override_or_discover(override_key, patterns):
    if CFG[override_key]:
        p = Path(CFG[override_key])
        assert p.exists(), f"Override path not found: {p}"
        return p
    # Try parquet first, then csv fallback
    found = latest_matching_file(TABLES_DIR, patterns)
    if found is None:
        csv_patterns = [p.replace(".parquet", ".csv") for p in patterns]
        found = latest_matching_file(TABLES_DIR, csv_patterns)
    return found

supervision_path  = _override_or_discover("supervision_table_path_override",
                        ["*training_supervision_table.parquet"])
splits_path       = _override_or_discover("splits_path_override",
                        ["*splits.parquet"])
feature_stats_path = _override_or_discover("feature_stats_path_override",
                        ["*feature_stats.parquet"])
union_path        = _override_or_discover("candidate_union_path_override",
                        ["*candidate_union.parquet"])

assert supervision_path and supervision_path.exists(), \
    "training_supervision_table not found — run NB05 first."
assert splits_path and splits_path.exists(), \
    "splits.parquet not found — run NB05 first."

log(f"  supervision : {supervision_path.name}")
log(f"  splits      : {splits_path.name}")
log(f"  feat_stats  : {feature_stats_path.name if feature_stats_path else 'not found'}")
log(f"  union       : {union_path.name if union_path else 'not found'}")

t0 = time.perf_counter()
if str(supervision_path).endswith(".csv"):
    log("[warn] Loading supervision table from CSV fallback")
    sup = pd.read_csv(supervision_path, low_memory=False)
else:
    sup = pd.read_parquet(supervision_path)

splits_df          = pd.read_parquet(splits_path)
feature_stats_nb04 = pd.read_parquet(feature_stats_path) if feature_stats_path and feature_stats_path.exists() else pd.DataFrame()
union_df           = pd.read_parquet(union_path) if union_path and union_path.exists() else pd.DataFrame()
log(f"  loaded in {time.perf_counter()-t0:.2f}s")
log(f"  supervision : {len(sup):,} rows × {len(sup.columns)} cols")
log(f"  splits      : {len(splits_df):,} rows")

In [ ]:
# -------------------------------------------------------------------
# SPLIT LOADING AND LEAKAGE VERIFICATION
# -------------------------------------------------------------------
log("=" * 90)
log("SPLIT VERIFICATION")

required_split_cols = {"split_id", "crop_id", "split_role", "split_registry_version"}
missing = sorted(required_split_cols - set(splits_df.columns))
assert not missing, f"splits.parquet missing columns: {missing}"
assert splits_df["crop_id"].is_unique, "Split leakage: crop_id in multiple splits"
assert set(splits_df["split_role"]).issubset({"train", "calibration", "test"})

# Supervised rows only — but keep crops with zero annotations as all-negative
# supervision_status == "included" covers both matched_positives and unmatched_negatives
# crops with no annotations produce zero supervised rows, which is correct —
# they only have excluded rows that were outside annotated territory
sup_labeled = sup[sup["supervision_status"] == "included"].copy()
sup_labeled["target_binary"] = sup_labeled["target_binary"].astype(float)

# Re-join split_role from canonical splits.parquet
sup_labeled = sup_labeled.drop(
    columns=[c for c in ["split_role", "split_id"] if c in sup_labeled.columns]
).merge(
    splits_df[["crop_id", "split_id", "split_role"]], on="crop_id", how="left"
)

# Crops with zero annotations won't be in sup_labeled at all — log them
annotated_crops  = set(sup[sup["supervision_status"] == "included"]["crop_id"].dropna().unique())
split_crops      = set(splits_df["crop_id"].unique())
zero_ann_crops   = split_crops - annotated_crops
if zero_ann_crops:
    log(f"[info] {len(zero_ann_crops)} crops in splits have zero annotations (no TRUE_SPOT clicks):")
    for c in sorted(zero_ann_crops):
        role = splits_df.loc[splits_df["crop_id"] == c, "split_role"].values
        log(f"  {c} -> {role[0] if len(role) else 'unknown'}")
    log("  These crops contribute no training rows — they are correctly handled.")
    log("  If they had candidates, those are 'excluded' (outside supervision territory).")

df_train = sup_labeled[sup_labeled["split_role"] == "train"].copy()
df_cal   = sup_labeled[sup_labeled["split_role"] == "calibration"].copy()
df_test  = sup_labeled[sup_labeled["split_role"] == "test"].copy()

# Leakage check
train_uids = set(df_train["union_id"])
cal_uids   = set(df_cal["union_id"])
test_uids  = set(df_test["union_id"])
assert len(train_uids & cal_uids)  == 0, "Leakage: union_id in train and cal"
assert len(train_uids & test_uids) == 0, "Leakage: union_id in train and test"
assert len(cal_uids   & test_uids) == 0, "Leakage: union_id in cal and test"
log("No split leakage ✓")

log(f"\nSplit sizes (supervised rows):")
for name, df in [("train", df_train), ("calibration", df_cal), ("test", df_test)]:
    if len(df) == 0:
        log(f"  {name:12s}: 0 rows (empty split)")
        continue
    n_pos = int(df["target_binary"].sum())
    n_neg = int(len(df) - n_pos)
    log(f"  {name:12s}: {len(df):5d} rows  pos={n_pos}  neg={n_neg}  pos_rate={n_pos/max(len(df),1):.4f}")
    if n_pos < 5:
        log(f"  [warn] {name} has only {n_pos} positives — metrics will be noisy")

assert len(df_train) > 0, "Training split is empty."
assert df_train["target_binary"].nunique() == 2, "Training split needs both classes."

In [ ]:
# -------------------------------------------------------------------
# FEATURE PREPARATION AND PRUNING
# All pruning decisions made on training rows only.
# -------------------------------------------------------------------
log("=" * 90)
log("FEATURE PREPARATION")

META_COLS = {
    "union_id", "dataset_id", "well_id", "crop_id", "cluster_id",
    "split_id", "split_role", "supervision_status", "target_binary",
    "target_source", "sample_weight", "match_status", "annotation_id",
    "gt_distance_px", "gt_label", "well_y_px", "well_x_px",
    "union_centroid_well_y_px", "union_centroid_well_x_px",
    "in_annotated_crop", "annotation_coverage_status",
    "is_completed_for_negatives", "eligible_for_negative_supervision",
    "coverage_allows_negative", "annotator_status", "crop_registry_version",
    "TRUE_SPOT", "UNCERTAIN", "run_id", "nb04_run_id",
    "project_config_version", "feature_registry_version", "created_at_utc",
}

all_feature_cols = [
    c for c in sup_labeled.columns
    if c not in META_COLS and pd.api.types.is_numeric_dtype(sup_labeled[c])
]
log(f"Candidate features before pruning: {len(all_feature_cols)}")

X_train_all = df_train[all_feature_cols]
y_train     = df_train["target_binary"].values
w_train     = df_train["sample_weight"].values

# ── Step 1: Drop all-NaN, constant, near-constant ─────────────────────────
null_frac   = X_train_all.isna().mean()
const_mask  = X_train_all.nunique() <= 1
nc_freq     = X_train_all.apply(lambda s: s.value_counts(normalize=True).iloc[0] if s.dropna().nunique() > 0 else 1.0)

keep1 = [
    c for c in all_feature_cols
    if null_frac[c] <= (1 - CFG["min_non_null_frac"])
    and not const_mask[c]
    and nc_freq[c] < CFG["near_constant_freq_threshold"]
]
log(f"  After null/constant/near-constant: {len(keep1)} kept ({len(all_feature_cols)-len(keep1)} dropped)")

# ── Step 2: Compute per-feature clip bounds on training data ──────────────
clip_bounds = {}
for c in keep1:
    vals = pd.to_numeric(X_train_all[c], errors="coerce").dropna()
    if len(vals):
        clip_bounds[c] = (float(vals.quantile(1 - CFG["clip_quantile"])),
                          float(vals.quantile(CFG["clip_quantile"])))
    else:
        clip_bounds[c] = (np.nan, np.nan)

def apply_clip(df: pd.DataFrame, features: list) -> pd.DataFrame:
    X = df[features].copy()
    for c in features:
        lo, hi = clip_bounds.get(c, (np.nan, np.nan))
        if np.isfinite(lo) and np.isfinite(hi):
            X[c] = X[c].clip(lower=lo, upper=hi)
    return X

# ── Step 3: Univariate ROC-AUC (reflected, train only) ───────────────────
log("  Computing univariate AUC per feature ...")
uni_aucs = {}
X_train_clipped = apply_clip(X_train_all, keep1)
for col in keep1:
    vals = X_train_clipped[col].fillna(X_train_clipped[col].median())
    if vals.nunique() < 2:
        uni_aucs[col] = 0.5; continue
    try:
        auc = roc_auc_score(y_train, vals)
        uni_aucs[col] = max(auc, 1 - auc)
    except:
        uni_aucs[col] = 0.5

keep2 = [c for c in keep1 if uni_aucs.get(c, 0.5) >= CFG["min_univariate_auc"]]
log(f"  After univariate AUC filter: {len(keep2)} kept")

# ── Step 4: Correlation filter ────────────────────────────────────────────
log("  Running correlation filter ...")
X_corr = X_train_clipped[keep2].fillna(X_train_clipped[keep2].median())
corr_matrix = X_corr.corr().abs()
dropped_corr = set()
for i, ci in enumerate(sorted(keep2, key=lambda c: uni_aucs.get(c, 0.5), reverse=True)):
    if ci in dropped_corr: continue
    for cj in sorted(keep2, key=lambda c: uni_aucs.get(c, 0.5), reverse=True)[i+1:]:
        if cj in dropped_corr: continue
        if ci in corr_matrix.index and cj in corr_matrix.columns:
            if corr_matrix.loc[ci, cj] > CFG["correlation_threshold"]:
                dropped_corr.add(cj)

FEATURE_COLS = [c for c in keep2 if c not in dropped_corr]
log(f"  After correlation filter: {len(FEATURE_COLS)} core_frozen features")
assert len(FEATURE_COLS) > 0, "All features pruned — adjust thresholds."

# ── Assemble feature matrices ──────────────────────────────────────────────
train_medians = df_train[FEATURE_COLS].median()

def get_X(df: pd.DataFrame) -> np.ndarray:
    X = apply_clip(df, FEATURE_COLS)
    return X.fillna(train_medians).values.astype(np.float32)

X_train = get_X(df_train)
X_cal   = get_X(df_cal)
X_test  = get_X(df_test)
y_cal   = df_cal["target_binary"].values
y_test  = df_test["target_binary"].values
w_cal   = df_cal["sample_weight"].values
w_test  = df_test["sample_weight"].values

log(f"  X_train={X_train.shape}  X_cal={X_cal.shape}  X_test={X_test.shape}")

In [ ]:
# -------------------------------------------------------------------
# CALIBRATION HELPER
# Fits both sigmoid and isotonic; winner chosen by Brier score.
# -------------------------------------------------------------------
def calibrate_model(fitted_model, X_cal, y_cal, X_test, tol=0.002):
    results = {}
    for method in ["sigmoid", "isotonic"]:
        try:
            from sklearn.frozen import FrozenEstimator
            frozen = FrozenEstimator(fitted_model)
        except ImportError:
            frozen = fitted_model
        cal = CalibratedClassifierCV(estimator=frozen, method=method, cv="prefit")
        cal.fit(X_cal, y_cal)
        brier = brier_score_loss(y_test, cal.predict_proba(X_test)[:, 1])
        results[method] = (cal, brier)
    b_sig, b_iso = results["sigmoid"][1], results["isotonic"][1]
    chosen = "isotonic" if b_iso < b_sig - tol else "sigmoid"
    return results[chosen][0], chosen, b_sig, b_iso

log("Calibration helper defined.")

In [ ]:
# -------------------------------------------------------------------
# STAGE 1 — BASELINE MODELS
# Threshold selected on CALIBRATION split (never test).
# -------------------------------------------------------------------
log("=" * 90)
log("STAGE 1: BASELINE MODELS")

RESULTS = {}  # model_id -> result dict

def fit_and_eval(model_id, raw_model, label=""):
    log(f"\n  Fitting {model_id} ...")
    t0 = time.perf_counter()
    if isinstance(raw_model, Pipeline):
        raw_model.fit(X_train, y_train,
                      **{f"{list(raw_model.named_steps.keys())[-1]}__sample_weight": w_train})
    else:
        raw_model.fit(X_train, y_train, sample_weight=w_train)
    log(f"  fit: {time.perf_counter()-t0:.1f}s")

    cal_model, cal_method, b_sig, b_iso = calibrate_model(
        raw_model, X_cal, y_cal, X_test, tol=CFG["calibration_tol"]
    )
    log(f"  calibration: sig={b_sig:.4f} iso={b_iso:.4f} chosen={cal_method}")

    prob_raw  = raw_model.predict_proba(X_test)[:, 1]
    prob_cal  = cal_model.predict_proba(X_test)[:, 1]
    prob_cal_cal = cal_model.predict_proba(X_cal)[:, 1]  # on cal split for threshold

    # ── Threshold on CALIBRATION split ────────────────────────────────────
    thr_info = choose_threshold(
        y_cal, prob_cal_cal,
        policy=CFG["threshold_policy"],
        fbeta=CFG["threshold_fbeta_beta"],
        precision_floor=CFG["threshold_precision_floor"],
        recall_floor=CFG["threshold_recall_floor"],
    )
    threshold = thr_info["threshold"]
    log(f"  threshold (on cal split, policy={CFG['threshold_policy']}): {threshold:.4f}")

    # ── Evaluate on TEST split ─────────────────────────────────────────────
    metrics_test = eval_metrics(y_test, prob_cal, threshold=threshold)

    # Also log train/cal metrics for overfitting check
    prob_cal_train = cal_model.predict_proba(X_train)[:, 1]
    metrics_train  = eval_metrics(y_train, prob_cal_train, threshold=threshold)
    metrics_cal    = eval_metrics(y_cal,   prob_cal_cal,   threshold=threshold)

    RESULTS[model_id] = {
        "model_raw":      raw_model,
        "model_cal":      cal_model,
        "cal_method":     cal_method,
        "prob_raw":       prob_raw,
        "prob_cal":       prob_cal,
        "prob_cal_cal":   prob_cal_cal,
        "prob_cal_train": prob_cal_train,
        "threshold":      threshold,
        "metrics_test":   metrics_test,
        "metrics_cal":    metrics_cal,
        "metrics_train":  metrics_train,
        "brier_sig":      b_sig,
        "brier_iso":      b_iso,
    }
    return RESULTS[model_id]

fit_and_eval("gbdt_baseline", GradientBoostingClassifier(**CFG["gbdt_baseline_params"]))
fit_and_eval("rf_baseline",   RandomForestClassifier(**CFG["rf_baseline_params"]))
fit_and_eval("lr_baseline",   Pipeline([("scaler", StandardScaler()),
                                         ("lr", LogisticRegression(**CFG["lr_baseline_params"]))]))

log("\nBaseline comparison (test split, threshold selected on cal):")
rows = []
for mid, res in RESULTS.items():
    m = res["metrics_test"]
    rows.append({"model_id": mid, "cal_method": res["cal_method"],
                 **{k: round(v, 4) for k, v in m.items()}})
display(pd.DataFrame(rows).set_index("model_id"))

In [ ]:
# -------------------------------------------------------------------
# TRAIN / CAL / TEST METRIC COMPARISON (overfitting check)
# -------------------------------------------------------------------
log("\nTrain vs Cal vs Test (AP and F1) — check for overfitting:")
ov_rows = []
for mid, res in RESULTS.items():
    ov_rows.append({
        "model_id": mid,
        "AP_train": round(res["metrics_train"]["avg_precision"], 4),
        "AP_cal":   round(res["metrics_cal"]["avg_precision"],   4),
        "AP_test":  round(res["metrics_test"]["avg_precision"],  4),
        "F1_train": round(res["metrics_train"]["f1"], 4),
        "F1_cal":   round(res["metrics_cal"]["f1"],   4),
        "F1_test":  round(res["metrics_test"]["f1"],  4),
    })
display(pd.DataFrame(ov_rows).set_index("model_id"))

In [ ]:
# -------------------------------------------------------------------
# STAGE 2 — OPTUNA HYPERPARAMETER SEARCH  [TOGGLE: run_optuna]
# GroupKFold spatial CV within training split only.
# Cal and test splits are never seen during search.
# Threshold still selected on cal split after refitting.
# -------------------------------------------------------------------
if CFG["run_optuna"] and OPTUNA_AVAILABLE:
    log("=" * 90)
    log("STAGE 2: OPTUNA HYPERPARAMETER SEARCH")

    # GroupKFold using crop_id as the spatial group
    groups_train   = df_train["crop_id"].values
    n_unique_crops = len(np.unique(groups_train))
    n_folds        = min(CFG["optuna_n_cv_folds"], n_unique_crops)

    if n_unique_crops < CFG["optuna_min_train_crops"]:
        log(f"[warn] Only {n_unique_crops} train crops < min {CFG['optuna_min_train_crops']}. Skipping Optuna.")
    else:
        if n_unique_crops >= n_folds:
            cv_splitter = GroupKFold(n_splits=n_folds)
            cv_splits   = list(cv_splitter.split(X_train, y_train, groups=groups_train))
        else:
            cv_splits = list(StratifiedKFold(n_splits=n_folds, shuffle=True,
                              random_state=CFG["random_seed"]).split(X_train, y_train))

        # Filter to folds with both classes in val
        cv_splits = [(tr, va) for tr, va in cv_splits
                     if len(np.unique(y_train[va])) == 2]
        log(f"Valid GroupKFold splits: {len(cv_splits)} / {n_folds}")

        def optuna_objective(trial):
            params = {
                "n_estimators":     CFG["optuna_max_trees_search"],
                "max_depth":        trial.suggest_int("max_depth", 2, 5),
                "learning_rate":    trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
                "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
                "max_features":     trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5]),
                "random_state":     CFG["random_seed"],
            }
            fold_scores = []
            for tr_idx, va_idx in cv_splits:
                Xtr, ytr, wtr = X_train[tr_idx], y_train[tr_idx], w_train[tr_idx]
                Xva, yva     = X_train[va_idx], y_train[va_idx]
                if len(np.unique(ytr)) < 2: continue
                m = GradientBoostingClassifier(**params)
                m.fit(Xtr, ytr, sample_weight=wtr)
                prob = m.predict_proba(Xva)[:, 1]
                if len(np.unique(yva)) >= 2:
                    fold_scores.append(average_precision_score(yva, prob))
            return float(np.mean(fold_scores)) if fold_scores else 0.0

        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(multivariate=True, seed=CFG["random_seed"]),
        )
        study.enqueue_trial({"max_depth": 4, "learning_rate": 0.05,
                              "subsample": 0.8, "min_samples_leaf": 5, "max_features": "sqrt"})

        log(f"Running {CFG['optuna_n_trials']} trials ...")
        t0 = time.perf_counter()
        study.optimize(optuna_objective, n_trials=CFG["optuna_n_trials"],
                       timeout=CFG["optuna_timeout_sec"], show_progress_bar=True)
        log(f"Done in {time.perf_counter()-t0:.1f}s  Best CV AP={study.best_value:.4f}")
        log(f"Best params: {study.best_params}")

        best_params = {**study.best_params, "n_estimators": 500, "random_state": CFG["random_seed"]}
        fit_and_eval("gbdt_tuned", GradientBoostingClassifier(**best_params))
        RESULTS["gbdt_tuned"]["best_cv_score"] = study.best_value
        RESULTS["gbdt_tuned"]["best_params"]   = best_params

        log("\nBaseline vs tuned:")
        rows = []
        for mid in ["gbdt_baseline", "gbdt_tuned"]:
            m = RESULTS[mid]["metrics_test"]
            rows.append({"model_id": mid, **{k: round(v,4) for k,v in m.items()}})
        display(pd.DataFrame(rows).set_index("model_id"))
else:
    if CFG["run_optuna"] and not OPTUNA_AVAILABLE:
        log("Optuna not installed — skipping. pip install optuna")
    else:
        log("Optuna disabled (CFG['run_optuna']=False). Enable when you have 50+ positives per split.")

In [ ]:
# -------------------------------------------------------------------
# STAGE 3 — LightGBM / XGBoost / CatBoost COMPARISON  [TOGGLE: run_lightgbm_family]
# Each model family gets its own Optuna search with GroupKFold CV.
# -------------------------------------------------------------------
if CFG["run_lightgbm_family"]:
    log("=" * 90)
    log("STAGE 3: LIGHTGBM FAMILY COMPARISON")

    if not OPTUNA_AVAILABLE:
        log("[warn] optuna required for LGB family tuning. pip install optuna")
    else:
        groups_train = df_train["crop_id"].values
        n_folds = min(CFG["optuna_n_cv_folds"], len(np.unique(groups_train)))
        cv_splits_lgb = []
        for tr, va in GroupKFold(n_splits=n_folds).split(X_train, y_train, groups=groups_train):
            if len(np.unique(y_train[va])) == 2:
                cv_splits_lgb.append((tr, va))

        def _lgb_objective(trial):
            if not LGB_AVAILABLE: return 0.0
            params = {
                "n_estimators":    trial.suggest_int("n_estimators", 100, 600),
                "max_depth":       trial.suggest_int("max_depth", 3, 8),
                "learning_rate":   trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "num_leaves":      trial.suggest_int("num_leaves", 15, 127),
                "subsample":       trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "reg_lambda":      trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
                "random_state":    CFG["random_seed"], "verbosity": -1,
            }
            scores = []
            for tr_idx, va_idx in cv_splits_lgb:
                m = lgb.LGBMClassifier(**params)
                m.fit(X_train[tr_idx], y_train[tr_idx], sample_weight=w_train[tr_idx],
                      eval_set=[(X_train[va_idx], y_train[va_idx])],
                      callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
                prob = m.predict_proba(X_train[va_idx])[:, 1]
                if len(np.unique(y_train[va_idx])) >= 2:
                    scores.append(average_precision_score(y_train[va_idx], prob))
            return float(np.mean(scores)) if scores else 0.0

        if LGB_AVAILABLE:
            log("Tuning LightGBM ...")
            lgb_study = optuna.create_study(direction="maximize",
                sampler=optuna.samplers.TPESampler(multivariate=True, seed=CFG["random_seed"]))
            lgb_study.optimize(_lgb_objective, n_trials=CFG["lgb_optuna_trials"], show_progress_bar=True)
            best_lgb_params = {**lgb_study.best_params, "verbosity": -1, "random_state": CFG["random_seed"]}
            lgb_model = lgb.LGBMClassifier(**best_lgb_params)
            fit_and_eval("lgbm_tuned", lgb_model)
            RESULTS["lgbm_tuned"]["best_cv_score"] = lgb_study.best_value

        log("\nAll models comparison (test split):")
        rows = []
        for mid, res in RESULTS.items():
            m = res["metrics_test"]
            rows.append({"model_id": mid, "cal_method": res["cal_method"],
                         **{k: round(v,4) for k,v in m.items()}})
        display(pd.DataFrame(rows).set_index("model_id"))
else:
    log("LightGBM family comparison disabled (CFG['run_lightgbm_family']=False).")

In [ ]:
# -------------------------------------------------------------------
# CHOOSE PRIMARY MODEL
# Best by avg_precision on test split.
# Override by setting CFG key or manually reassigning PRIMARY_MODEL_ID.
# -------------------------------------------------------------------
PRIMARY_MODEL_ID = max(RESULTS, key=lambda mid: RESULTS[mid]["metrics_test"].get("avg_precision", 0))
primary = RESULTS[PRIMARY_MODEL_ID]
log(f"Primary model: {PRIMARY_MODEL_ID}  (AP={primary['metrics_test']['avg_precision']:.4f})")
log(f"  threshold={primary['threshold']:.4f}  (selected on cal split, policy={CFG['threshold_policy']})")

In [ ]:
# -------------------------------------------------------------------
# EVALUATION ON TEST SPLIT — full metric suite + stratified breakdowns
# -------------------------------------------------------------------
log("=" * 90)
log("EVALUATION ON TEST SPLIT")

# Full model comparison
log("\nFull comparison (test split, threshold from cal):")
summary_rows = []
for mid, res in RESULTS.items():
    m = res["metrics_test"]
    summary_rows.append({"model_id": mid, "cal_method": res["cal_method"],
                          "threshold": round(res["threshold"], 4),
                          **{k: round(v,4) for k,v in m.items()}})
metric_comparison_df = pd.DataFrame(summary_rows).set_index("model_id")
display(metric_comparison_df)

# Attach predictions to test DataFrame
df_test_eval = df_test.copy()
df_test_eval["prob_cal"]  = primary["prob_cal"]
df_test_eval["prob_raw"]  = primary["prob_raw"]
df_test_eval["predicted"] = (df_test_eval["prob_cal"] >= primary["threshold"]).astype(int)

# Spatial stratum classification
if "well_y_px" in df_test_eval.columns and "well_x_px" in df_test_eval.columns:
    coords = df_test_eval[["well_y_px", "well_x_px"]].fillna(0).values
    if len(coords) > 1:
        tree   = cKDTree(coords)
        counts = np.array([len(tree.query_ball_point(c, r=CFG["isolation_radius_px"])) - 1 for c in coords])
    else:
        counts = np.zeros(len(coords), dtype=int)
    df_test_eval["n_neighbors_iso"] = counts
    df_test_eval["spatial_stratum"] = np.where(
        counts == 0, "isolated", np.where(counts == 1, "close_pair", "cluster")
    )
else:
    df_test_eval["spatial_stratum"] = "unknown"

# Error type
df_test_eval["error_type"] = "TN"
df_test_eval.loc[(df_test_eval["target_binary"]==1)&(df_test_eval["predicted"]==1), "error_type"] = "TP"
df_test_eval.loc[(df_test_eval["target_binary"]==1)&(df_test_eval["predicted"]==0), "error_type"] = "FN"
df_test_eval.loc[(df_test_eval["target_binary"]==0)&(df_test_eval["predicted"]==1), "error_type"] = "FP"

# Stratified breakdown
log("\nStratified breakdown (primary model, test split):")
strat_rows = []
for stratum, g in df_test_eval.groupby("spatial_stratum"):
    gyt = g["target_binary"].values; gyp = g["prob_cal"].values
    if len(np.unique(gyt)) < 2 or len(gyt) < 2: continue
    m = eval_metrics(gyt, gyp, threshold=primary["threshold"])
    strat_rows.append({"stratum": stratum, "n": len(g), **{k: round(v,4) for k,v in m.items()}})
if strat_rows:
    display(pd.DataFrame(strat_rows).set_index("stratum"))

# Per-well and per-crop
log("\nPer-well metrics:")
for well_id, g in df_test_eval.groupby("well_id"):
    gyt = g["target_binary"].values; gyp = g["prob_cal"].values
    if len(np.unique(gyt)) < 2: continue
    m = eval_metrics(gyt, gyp, threshold=primary["threshold"])
    log(f"  {well_id}: AP={m['avg_precision']:.3f}  F1={m['f1']:.3f}  n={len(g)}")

# Error taxonomy
log("\nError type counts:")
display(df_test_eval["error_type"].value_counts().to_frame("n"))
if "spatial_stratum" in df_test_eval.columns:
    log("\nErrors by spatial stratum:")
    display(df_test_eval.groupby(["spatial_stratum","error_type"]).size().unstack(fill_value=0))

# SNR proxy
snr_col = next((c for c in ["photo_norm01_inner_mean","resp_log_normalized_center",
                              "photo_raw_inner_mean"] if c in df_test_eval.columns), None)
if snr_col:
    q = df_test_eval[snr_col].quantile([0.33, 0.67])
    df_test_eval["snr_bin"] = pd.cut(df_test_eval[snr_col],
        bins=[-np.inf, float(q.iloc[0]), float(q.iloc[1]), np.inf],
        labels=["low","medium","high"])
    log("\nErrors by SNR bin:")
    display(df_test_eval.groupby(["snr_bin","error_type"]).size().unstack(fill_value=0))

In [ ]:
# -------------------------------------------------------------------
# DIAGNOSTIC PLOTS — ROC, PR, Calibration
# -------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot([0,1],[0,1],"k--",alpha=0.4,label="random")
for mid, res in RESULTS.items():
    fpr, tpr, _ = roc_curve(y_test, res["prob_cal"])
    ax.plot(fpr, tpr, lw=1.5, label=f"{mid} (AUC={res['metrics_test']['roc_auc']:.3f})")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("ROC (test split)")
ax.legend(fontsize=7); ax.set_xlim(0,1); ax.set_ylim(0,1)

ax = axes[1]
ax.axhline(y_test.mean(), color="k", linestyle="--", alpha=0.4,
           label=f"random (AP={y_test.mean():.3f})")
for mid, res in RESULTS.items():
    prec, rec, _ = precision_recall_curve(y_test, res["prob_cal"])
    ax.plot(rec, prec, lw=1.5, label=f"{mid} (AP={res['metrics_test']['avg_precision']:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("PR (test split)")
ax.legend(fontsize=7); ax.set_xlim(0,1); ax.set_ylim(0,1)

ax = axes[2]
ax.plot([0,1],[0,1],"k--",alpha=0.4,label="perfect")
for mid, res in RESULTS.items():
    fp, mp = calibration_curve(y_test, res["prob_cal"], n_bins=10, strategy="quantile")
    ax.plot(mp, fp, "o-", markersize=4, lw=1.5,
            label=f"{mid} (Brier={res['metrics_test']['brier']:.3f})")
ax.set_xlabel("Mean predicted"); ax.set_ylabel("Fraction positives")
ax.set_title("Reliability diagram\n(quantile bins — ECE unreliable under imbalance)")
ax.legend(fontsize=7); ax.set_xlim(0,1); ax.set_ylim(0,1)

fig.tight_layout()
if CFG["write_reports"]:
    p = REPORT_DIR / f"{RUN_ID}_roc_pr_cal.png"
    fig.savefig(p, dpi=150, bbox_inches="tight")
    log(f"Saved: {p.name}")
plt.show(); plt.close(fig)

In [ ]:
# -------------------------------------------------------------------
# THRESHOLD SELECTION REPORT
# Sweep on test split for visualization only — threshold already chosen on cal.
# -------------------------------------------------------------------
log("=" * 90)
log("THRESHOLD SELECTION REPORT")

thresholds = np.linspace(0.01, 0.99, CFG["threshold_sweep_steps"])
sweep_rows = []
for thr in thresholds:
    y_pred = (primary["prob_cal"] >= thr).astype(int)
    sweep_rows.append({
        "threshold":       float(thr),
        "precision":       float(precision_score(y_test, y_pred, zero_division=0)),
        "recall":          float(recall_score(y_test, y_pred, zero_division=0)),
        "f1":              float(f1_score(y_test, y_pred, zero_division=0)),
        "n_predicted_pos": int(y_pred.sum()),
    })
threshold_sweep_df = pd.DataFrame(sweep_rows)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["precision"], label="Precision")
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["recall"],    label="Recall")
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["f1"], lw=2,  label="F1")
ax.axvline(primary["threshold"], color="red", linestyle="--",
           label=f"Chosen threshold ({primary['threshold']:.3f}, from cal split)")
ax.axvline(0.5, color="gray", linestyle=":", alpha=0.5, label="0.5")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score")
ax.set_title(f"Threshold sweep — {PRIMARY_MODEL_ID} (visualized on test split)\n"
             "Note: threshold was selected on calibration split to avoid optimistic bias")
ax.legend(fontsize=8); ax.set_xlim(0,1); ax.set_ylim(0,1)
fig.tight_layout()
if CFG["write_reports"]:
    p = REPORT_DIR / f"{RUN_ID}_threshold_sweep.png"
    fig.savefig(p, dpi=150, bbox_inches="tight")
plt.show(); plt.close(fig)
display(threshold_sweep_df.head(10))

In [ ]:
# -------------------------------------------------------------------
# FEATURE IMPORTANCE — MDI + Permutation  [TOGGLE: run_permutation_importance]
# -------------------------------------------------------------------
log("=" * 90)
log("FEATURE IMPORTANCE")

primary_raw = primary["model_raw"]
if hasattr(primary_raw, "named_steps"):
    primary_raw_model = primary_raw.named_steps[list(primary_raw.named_steps)[-1]]
else:
    primary_raw_model = primary_raw

mdi_df = pd.DataFrame()
if hasattr(primary_raw_model, "feature_importances_"):
    mdi_df = pd.DataFrame({
        "feature":        FEATURE_COLS,
        "mdi_importance": primary_raw_model.feature_importances_,
        "univariate_auc": [uni_aucs.get(c, np.nan) for c in FEATURE_COLS],
    }).sort_values("mdi_importance", ascending=False).reset_index(drop=True)
    log("\nTop 20 by MDI importance:")
    display(mdi_df.head(20))

perm_df = pd.DataFrame()
if CFG["run_permutation_importance"]:
    log("\nComputing permutation importance ...")
    try:
        t0 = time.perf_counter()
        perm_result = sklearn_perm_importance(
            primary_raw_model, X_test, y_test,
            n_repeats=CFG["permutation_n_repeats"],
            random_state=CFG["random_seed"],
            scoring="average_precision", n_jobs=-1,
        )
        log(f"  done in {time.perf_counter()-t0:.1f}s")
        perm_df = pd.DataFrame({
            "feature":    FEATURE_COLS,
            "perm_mean":  perm_result.importances_mean,
            "perm_std":   perm_result.importances_std,
        }).sort_values("perm_mean", ascending=False).reset_index(drop=True)
        log("Top 20 by permutation importance (AP drop):")
        display(perm_df.head(20))
    except Exception as e:
        log(f"[warn] Permutation importance failed: {e}")

In [ ]:
# -------------------------------------------------------------------
# SHAP ANALYSIS  [TOGGLE: run_shap]
# -------------------------------------------------------------------
shap_df = pd.DataFrame()
shap_values_arr = None

if CFG["run_shap"] and SHAP_AVAILABLE and hasattr(primary_raw_model, "feature_importances_"):
    log("=" * 90)
    log("SHAP ANALYSIS")

    n_shap   = min(CFG["shap_max_rows"], len(X_test))
    rng      = np.random.default_rng(CFG["random_seed"])
    shap_idx = rng.choice(len(X_test), size=n_shap, replace=False)
    X_shap   = X_test[shap_idx]
    y_shap   = y_test[shap_idx]

    try:
        t0 = time.perf_counter()
        explainer       = shap.TreeExplainer(primary_raw_model)
        shap_values_arr = explainer.shap_values(X_shap)
        if isinstance(shap_values_arr, list):
            shap_values_arr = shap_values_arr[1] if len(shap_values_arr) > 1 else shap_values_arr[0]
        log(f"  TreeSHAP done in {time.perf_counter()-t0:.1f}s")

        mean_abs = np.abs(shap_values_arr).mean(axis=0)
        shap_df  = pd.DataFrame({
            "feature":       FEATURE_COLS,
            "shap_mean_abs": mean_abs,
            "univariate_auc": [uni_aucs.get(c, np.nan) for c in FEATURE_COLS],
        }).sort_values("shap_mean_abs", ascending=False).reset_index(drop=True)

        log(f"\nTop 25 by SHAP mean |value| (n={n_shap} test rows):")
        display(shap_df.head(25))

        # Bar plot
        fig, ax = plt.subplots(figsize=(9, 8))
        shap.summary_plot(shap_values_arr, X_shap, feature_names=FEATURE_COLS,
                          plot_type="bar", max_display=25, show=False)
        plt.title(f"SHAP mean |value| — {PRIMARY_MODEL_ID} (n={n_shap})")
        plt.tight_layout()
        if CFG["write_reports"]:
            plt.savefig(REPORT_DIR / f"{RUN_ID}_shap_bar.png", dpi=150, bbox_inches="tight")
        plt.show(); plt.close()

        # Beeswarm
        fig, ax = plt.subplots(figsize=(9, 10))
        shap.summary_plot(shap_values_arr, X_shap, feature_names=FEATURE_COLS,
                          max_display=25, show=False)
        plt.title(f"SHAP beeswarm — {PRIMARY_MODEL_ID}\n"
                  "Red=high feature value, Blue=low. Right=pushes toward TRUE_SPOT.")
        plt.tight_layout()
        if CFG["write_reports"]:
            plt.savefig(REPORT_DIR / f"{RUN_ID}_shap_beeswarm.png", dpi=150, bbox_inches="tight")
        plt.show(); plt.close()

        # Dependence plots for top 4
        top4 = shap_df["feature"].head(4).tolist()
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        for ax, feat in zip(axes.flat, top4):
            feat_idx = FEATURE_COLS.index(feat)
            shap.dependence_plot(feat_idx, shap_values_arr, X_shap,
                                 feature_names=FEATURE_COLS, ax=ax, show=False)
            ax.set_title(feat, fontsize=9)
        fig.suptitle(f"SHAP dependence — top 4 features | {PRIMARY_MODEL_ID}", y=1.01)
        fig.tight_layout()
        if CFG["write_reports"]:
            plt.savefig(REPORT_DIR / f"{RUN_ID}_shap_dependence.png", dpi=150, bbox_inches="tight")
        plt.show(); plt.close()

        # Waterfall for TP, FP, FN examples
        df_shap_meta = df_test.iloc[shap_idx].copy().reset_index(drop=True)
        df_shap_meta["prob_cal"]  = primary["prob_cal"][shap_idx]
        df_shap_meta["predicted"] = (df_shap_meta["prob_cal"] >= primary["threshold"]).astype(int)
        df_shap_meta["error_type"] = "TN"
        df_shap_meta.loc[(df_shap_meta["target_binary"]==1)&(df_shap_meta["predicted"]==1), "error_type"] = "TP"
        df_shap_meta.loc[(df_shap_meta["target_binary"]==1)&(df_shap_meta["predicted"]==0), "error_type"] = "FN"
        df_shap_meta.loc[(df_shap_meta["target_binary"]==0)&(df_shap_meta["predicted"]==1), "error_type"] = "FP"

        ev = explainer.expected_value
        if isinstance(ev, (list, np.ndarray)):
            ev = ev[1] if len(ev) > 1 else ev[0]

        for error_type in ["TP", "FP", "FN"]:
            rows = df_shap_meta[df_shap_meta["error_type"] == error_type]
            if len(rows) == 0:
                log(f"  No {error_type} in SHAP sample — skipping waterfall")
                continue
            local_idx = rows.index[0]
            expl_obj  = shap.Explanation(
                values=shap_values_arr[local_idx],
                base_values=float(ev),
                data=X_shap[local_idx],
                feature_names=FEATURE_COLS,
            )
            fig, ax = plt.subplots(figsize=(10, 7))
            shap.waterfall_plot(expl_obj, max_display=15, show=False)
            plt.title(f"SHAP waterfall — {error_type} example  "
                      f"(prob={df_shap_meta.iloc[local_idx]['prob_cal']:.4f})", fontsize=10)
            plt.tight_layout()
            if CFG["write_reports"]:
                plt.savefig(REPORT_DIR / f"{RUN_ID}_shap_waterfall_{error_type}.png",
                            dpi=150, bbox_inches="tight")
            plt.show(); plt.close()

    except Exception as e:
        log(f"[warn] SHAP failed: {type(e).__name__}: {e}")
else:
    if CFG["run_shap"] and not SHAP_AVAILABLE:
        log("SHAP skipped — not installed. pip install shap")
    elif not CFG["run_shap"]:
        log("SHAP disabled (CFG['run_shap']=False).")

In [ ]:
# -------------------------------------------------------------------
# ENRICH feature_stats.parquet (§3.12)
# -------------------------------------------------------------------
log("=" * 90)
log("ENRICHING feature_stats")

shap_lookup  = dict(zip(shap_df["feature"],  shap_df["shap_mean_abs"])) if len(shap_df) else {}
perm_lookup  = dict(zip(perm_df["feature"],  perm_df["perm_mean"]))     if len(perm_df) else {}
mdi_lookup   = dict(zip(mdi_df["feature"],   mdi_df["mdi_importance"])) if len(mdi_df) else {}

fs_rows = []
for col in all_feature_cols:
    in_core = col in FEATURE_COLS
    group   = _infer_group(col)
    if len(feature_stats_nb04) > 0 and "feature_name" in feature_stats_nb04.columns:
        nb04_row = feature_stats_nb04[feature_stats_nb04["feature_name"] == col]
        if len(nb04_row) and "feature_group" in nb04_row.columns:
            group = nb04_row.iloc[0]["feature_group"]
    fs_rows.append({
        "feature_name":            col,
        "feature_group":           group,
        "is_core_frozen":          in_core,
        "non_null_fraction":       float(1 - null_frac.get(col, 1.0)),
        "constant_flag":           bool(const_mask.get(col, False)),
        "univariate_score":        float(uni_aucs.get(col, np.nan)),
        "correlation_pruned_flag": col in dropped_corr,
        "mdi_importance":          float(mdi_lookup.get(col, np.nan)),
        "shap_mean_abs":           float(shap_lookup.get(col, np.nan)),
        "perm_importance_mean":    float(perm_lookup.get(col, np.nan)),
        "retained_for_model":      in_core,
        "model_registry_version":  CFG["model_registry_version"],
    })

feature_stats_nb06 = pd.DataFrame(fs_rows)
log(f"feature_stats: {len(feature_stats_nb06)} features  "
    f"core_frozen={feature_stats_nb06['is_core_frozen'].sum()}")

log("\nFeature group distribution (core_frozen):")
display(
    feature_stats_nb06[feature_stats_nb06["is_core_frozen"]]
    .groupby("feature_group").size().rename("n").sort_values(ascending=False)
)

log("\nTop 20 core features by univariate AUC:")
display(
    feature_stats_nb06[feature_stats_nb06["is_core_frozen"]]
    .sort_values("univariate_score", ascending=False).head(20)
)

In [ ]:
# -------------------------------------------------------------------
# BUILD model_predictions.parquet (§3.11)
# Includes train, cal, and test splits for completeness.
# -------------------------------------------------------------------
log("=" * 90)
log("BUILDING model_predictions")

pred_rows = []
for mid, res in RESULTS.items():
    # test split predictions
    for i, uid in enumerate(df_test["union_id"].values):
        pred_rows.append({
            "prediction_id":             sha1_text(f"{uid}|{mid}|test|{RUN_ID}"),
            "union_id":                  uid,
            "model_id":                  mid,
            "model_version":             CFG["model_registry_version"],
            "split_id":                  df_test.iloc[i].get("split_id", None),
            "split_role":                "test",
            "prob_true_spot":            float(res["prob_raw"][i]),
            "prob_true_spot_calibrated": float(res["prob_cal"][i]),
            "decision_threshold":        float(res["threshold"]),
            "predicted_label":           int(res["prob_cal"][i] >= res["threshold"]),
            "calibration_method":        res["cal_method"],
            "threshold_policy":          CFG["threshold_policy"],
            "run_id":                    RUN_ID,
            "created_at_utc":            datetime.now(timezone.utc).isoformat(),
        })

model_predictions = pd.DataFrame(pred_rows)
assert model_predictions["prediction_id"].is_unique, "prediction_id must be unique"
log(f"model_predictions: {len(model_predictions):,} rows")
display(model_predictions.head())

In [ ]:
# -------------------------------------------------------------------
# VISUAL QA OVERLAYS  [TOGGLE: run_visual_qa]
# Shows all available channels per crop.
# Handles .tiff files with naming: {WELL}_common_Fluorescence_{WL}_nm_Ex.tiff
# -------------------------------------------------------------------
if CFG["run_visual_qa"]:
    log("=" * 90)
    log("VISUAL QA OVERLAYS")

    # ── Find data root ─────────────────────────────────────────────────────
    _data_candidates = [
        REPO_ROOT / "data",
        Path.home() / "Desktop" / "bpa1",
        Path.home() / "Desktop" / "BPA1",
        Path.home() / "Desktop",
    ]
    DATA_ROOT = None
    for cand in _data_candidates:
        if cand.exists() and list(cand.rglob("*.tiff")):
            DATA_ROOT = cand; break
    if DATA_ROOT is None:
        log("[warn] No .tiff files found — skipping visual QA. Set DATA_ROOT manually.")
    else:
        log(f"DATA_ROOT: {DATA_ROOT}")

        # ── Build image file index ──────────────────────────────────────────
        # Pattern: {WELL}_common_Fluorescence_{WL}_nm_Ex.tiff  or similar
        _WL_RE   = re.compile(r"(\d{3})_nm", re.IGNORECASE)
        _WELL_RE = re.compile(r"(?<![A-Z0-9])([A-H](?:[1-9]|1[0-2]))(?![A-Z0-9])", re.IGNORECASE)

        WAVELENGTH_LABELS = {"405": "DAPI (405nm)", "561": "Ch561 (561nm)",
                             "638": "Ch638 (638nm)", "BF": "BF"}

        def parse_well_wl(fname):
            wm = _WELL_RE.search(fname)
            well = wm.group(1).upper() if wm else None
            # wavelength
            if "BF" in fname.upper():
                wl = "BF"
            else:
                wlm = _WL_RE.search(fname)
                wl = wlm.group(1) if wlm else None
            return well, wl

        img_rows = []
        for f in DATA_ROOT.rglob("*.tiff"):
            if f.name.startswith("._"): continue
            well, wl = parse_well_wl(f.name)
            if well and wl:
                img_rows.append({"well_id": well, "wavelength": wl, "path": f})
        for f in DATA_ROOT.rglob("*.tif"):
            if f.name.startswith("._"): continue
            well, wl = parse_well_wl(f.name)
            if well and wl:
                img_rows.append({"well_id": well, "wavelength": wl, "path": f})

        if not img_rows:
            log("[warn] No .tiff/.tif files parsed — check filename pattern in WAVELENGTH_LABELS above.")
        else:
            img_index = pd.DataFrame(img_rows)
            log(f"Found {len(img_index)} image files across {img_index['well_id'].nunique()} wells")
            log(f"Wavelengths found: {sorted(img_index['wavelength'].unique())}")

            def get_channel_projections(well_id):
                """Return dict: wavelength -> 2D float32 max-projection."""
                result = {}
                for _, row in img_index[img_index["well_id"] == well_id].iterrows():
                    wl = row["wavelength"]
                    if wl in result: continue  # keep first match per wavelength
                    try:
                        arr = np.asarray(tifffile.imread(str(row["path"]))).squeeze()
                        if arr.ndim == 3: arr = arr.max(axis=0)
                        result[wl] = arr.astype(np.float32)
                    except Exception as e:
                        log(f"  [warn] {well_id} wl={wl}: {e}")
                return result

            def pct_clip(img, lo=1, hi=99):
                lo_v = np.percentile(img, lo); hi_v = np.percentile(img, hi)
                return np.clip((img - lo_v) / max(hi_v - lo_v, 1e-8), 0, 1)

            # ── Load crop registry ──────────────────────────────────────────
            cr_dir   = REPO_ROOT / "annotations" / "crop_registry"
            cr_files = sorted(cr_dir.glob("*.parquet")) + sorted(cr_dir.glob("*.yaml"))
            if not cr_files:
                log("[warn] No crop registry found — skipping overlays")
            else:
                if cr_files[-1].suffix == ".parquet":
                    crop_reg = pd.read_parquet(cr_files[-1])
                else:
                    _pl = yaml.safe_load(cr_files[-1].read_text())
                    crop_reg = pd.DataFrame(_pl["records"] if "records" in _pl else _pl)
                crop_reg["crop_id"] = crop_reg["crop_id"].astype(str)

                # ── Load annotations ────────────────────────────────────────
                ann_path = TABLES_DIR / "annotations.parquet"
                ann = pd.read_parquet(ann_path) if ann_path.exists() else pd.DataFrame()
                if len(ann):
                    ann["crop_id"] = ann["crop_id"].astype(str)

                # ── Choose crops to display ──────────────────────────────────
                test_crops = df_test_eval["crop_id"].dropna().unique().tolist()
                all_ann_crops = ann["crop_id"].dropna().unique().tolist() if len(ann) else []
                priority = test_crops + [c for c in all_ann_crops if c not in test_crops]
                display_crops = [c for c in priority
                                 if c in crop_reg["crop_id"].values][:CFG["qa_n_crops"]]
                log(f"Rendering {len(display_crops)} crops (test-split first)")

                # Cache images per well
                well_cache = {}
                for cid in display_crops:
                    wid = str(crop_reg.loc[crop_reg["crop_id"]==cid, "well_id"].iloc[0])
                    if wid not in well_cache:
                        log(f"  Loading channels for well {wid} ...")
                        well_cache[wid] = get_channel_projections(wid)

                SHOW_WLS = sorted(
                    [wl for wl in WAVELENGTH_LABELS if wl != "BF"
                     and any(wl in well_cache[list(well_cache.keys())[0]]
                             for _ in [None])]
                ) if well_cache else []
                # Fallback: show all available wavelengths
                if not SHOW_WLS and well_cache:
                    first_well = list(well_cache.values())[0]
                    SHOW_WLS = sorted(first_well.keys())

                COLOR_MAP = {
                    "matched_positive":               "lime",
                    "unmatched_candidate_negative":   None,    # don't show negatives — too many
                    "excluded_uncertain":             "gold",
                    "unmatched_truth_false_negative": "cyan",
                }

                pred_lookup_all = df_test_eval.set_index("union_id") if len(df_test_eval) else pd.DataFrame()

                for crop_id in display_crops:
                    cr_row  = crop_reg[crop_reg["crop_id"] == crop_id].iloc[0]
                    well_id = str(cr_row["well_id"])
                    y0 = int(cr_row["well_ymin_px"]); y1 = int(cr_row["well_ymax_px"])
                    x0 = int(cr_row["well_xmin_px"]); x1 = int(cr_row["well_xmax_px"])

                    channels = well_cache.get(well_id, {})
                    wls_avail = [wl for wl in SHOW_WLS if wl in channels]
                    if not wls_avail:
                        wls_avail = [wl for wl in channels.keys() if wl != "BF"]
                    if not wls_avail:
                        log(f"  [skip] {crop_id}: no displayable channels for well {well_id}")
                        continue

                    n_panels = len(wls_avail)
                    fig, axes_row = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5.5),
                                                gridspec_kw={"wspace": 0.03})
                    if n_panels == 1: axes_row = [axes_row]

                    for ax, wl in zip(axes_row, wls_avail):
                        img  = channels[wl][y0:y1, x0:x1]
                        disp = pct_clip(img, CFG["qa_display_lo_pct"], CFG["qa_display_hi_pct"])
                        ax.imshow(disp, cmap="gray", origin="upper")
                        ax.set_title(WAVELENGTH_LABELS.get(wl, wl), fontsize=10)
                        ax.set_xticks([]); ax.set_yticks([])

                    # ── Draw annotations ──────────────────────────────────────
                    R = CFG["qa_circle_radius_px"]
                    if len(ann):
                        crop_ann   = ann[ann["crop_id"] == crop_id]
                        true_spots = crop_ann[crop_ann["label"] == "TRUE_SPOT"]
                        uncertain  = crop_ann[crop_ann["label"] == "UNCERTAIN"]
                        for ax in axes_row:
                            for _, a in true_spots.iterrows():
                                cy = float(a["refined_click_well_y_px"]) - y0
                                cx = float(a["refined_click_well_x_px"]) - x0
                                ax.add_patch(patches.Circle((cx,cy), R, lw=2,
                                    edgecolor="lime", facecolor="none", zorder=5))
                                ax.plot(cx, cy, ".", color="lime", ms=3, zorder=6)
                            for _, a in uncertain.iterrows():
                                cy = float(a["refined_click_well_y_px"]) - y0
                                cx = float(a["refined_click_well_x_px"]) - x0
                                ax.add_patch(patches.Circle((cx,cy), R, lw=1.5,
                                    edgecolor="orange", facecolor="none",
                                    linestyle="--", zorder=5))
                    else:
                        true_spots = uncertain = pd.DataFrame()

                    # ── Draw model predictions ────────────────────────────────
                    crop_cands = union_df[union_df["crop_id"] == crop_id] if len(union_df) else pd.DataFrame()
                    n_pred = n_tp = n_fp = n_fn = 0
                    for _, cand in crop_cands.iterrows():
                        uid = cand["union_id"]
                        cy  = float(cand["union_medoid_well_y_px"]) - y0
                        cx  = float(cand["union_medoid_well_x_px"]) - x0
                        H, W = y1-y0, x1-x0
                        if not (0 <= cy < H and 0 <= cx < W): continue

                        if len(pred_lookup_all) and uid in pred_lookup_all.index:
                            row_p   = pred_lookup_all.loc[uid]
                            prob    = float(row_p["prob_cal"]) if pd.notna(row_p["prob_cal"]) else None
                            pred    = int(row_p["predicted"]) if pd.notna(row_p["predicted"]) else None
                            etype   = row_p.get("error_type", None)
                        else:
                            prob = pred = etype = None

                        for ax in axes_row:
                            if pred == 1:
                                n_pred += 1
                                if etype == "TP":
                                    n_tp += 1
                                    ax.add_patch(patches.Circle((cx,cy), R+4, lw=2.5,
                                        edgecolor="red", facecolor="none", zorder=4))
                                elif etype == "FP":
                                    n_fp += 1
                                    ax.plot(cx, cy, "x", color="magenta", ms=14,
                                            markeredgewidth=2.5, zorder=6)
                                else:
                                    ax.add_patch(patches.Circle((cx,cy), R+4, lw=1.5,
                                        edgecolor="red", facecolor="none",
                                        linestyle="--", zorder=4))
                            elif etype == "FN":
                                n_fn += 1
                                ax.plot(cx, cy, "x", color="cyan", ms=14,
                                        markeredgewidth=2.5, zorder=6)

                    # ── Legend ────────────────────────────────────────────────
                    in_test = crop_id in df_test_eval["crop_id"].values
                    split_label = ("TEST" if in_test
                                   else "TRAIN/CAL" if crop_id in sup_labeled["crop_id"].values
                                   else "NO LABELS")
                    legend_elements = [
                        patches.Patch(facecolor="none", edgecolor="lime",    lw=2,   label=f"TRUE_SPOT annotation ({len(true_spots)})"),
                        patches.Patch(facecolor="none", edgecolor="orange",  lw=1.5, label=f"UNCERTAIN ({len(uncertain)})"),
                        patches.Patch(facecolor="none", edgecolor="red",     lw=2.5, label=f"Model predicted pos ({n_pred})"),
                        Line2D([0],[0], marker="x", color="magenta", linestyle="None", ms=10, markeredgewidth=2, label=f"FP ({n_fp})"),
                        Line2D([0],[0], marker="x", color="cyan",    linestyle="None", ms=10, markeredgewidth=2, label=f"FN ({n_fn})"),
                    ]
                    axes_row[-1].legend(handles=legend_elements, loc="upper right",
                                        fontsize=7, framealpha=0.85)
                    fig.suptitle(
                        f"{well_id}  |  {crop_id[-16:]}  [{split_label}]  "
                        f"y:[{y0},{y1})  x:[{x0},{x1})\n"
                        f"Annotations: {len(true_spots)} TRUE_SPOT  {len(uncertain)} UNCERTAIN  |  "
                        f"Model: {n_pred} predicted  TP={n_tp}  FP={n_fp}  FN={n_fn}",
                        fontsize=9, y=1.01
                    )
                    plt.tight_layout()
                    if CFG["write_reports"]:
                        p = REPORT_DIR / f"{RUN_ID}_qa_{well_id}_{crop_id[-12:]}.png"
                        plt.savefig(p, dpi=160, bbox_inches="tight")
                    plt.show(); plt.close(fig)

        log(f"Visual QA complete.")
else:
    log("Visual QA disabled (CFG['run_visual_qa']=False).")

In [ ]:
# -------------------------------------------------------------------
# PERSIST ALL OUTPUTS
# -------------------------------------------------------------------
log("=" * 90)
log("PERSISTING OUTPUTS")

output_paths = {}

if CFG["write_model_predictions"]:
    p = safe_to_parquet(model_predictions, TABLES_DIR / f"{RUN_ID}_model_predictions.parquet")
    output_paths["model_predictions"] = str(p); log(f"  {p.name}")

if CFG["write_feature_stats"]:
    p = safe_to_parquet(feature_stats_nb06, TABLES_DIR / f"{RUN_ID}_feature_stats.parquet")
    output_paths["feature_stats"] = str(p); log(f"  {p.name}")

if CFG["write_reports"]:
    p = safe_to_parquet(threshold_sweep_df, REPORT_DIR / f"{RUN_ID}_threshold_sweep.parquet")
    output_paths["threshold_sweep"] = str(p)
    p = safe_to_parquet(metric_comparison_df.reset_index(), REPORT_DIR / f"{RUN_ID}_model_comparison.parquet")
    output_paths["model_comparison"] = str(p)
    if len(shap_df):
        p = safe_to_parquet(shap_df, REPORT_DIR / f"{RUN_ID}_shap_importance.parquet")
        output_paths["shap_importance"] = str(p)
    if len(perm_df):
        p = safe_to_parquet(perm_df, REPORT_DIR / f"{RUN_ID}_perm_importance.parquet")
        output_paths["perm_importance"] = str(p)
    p = safe_to_parquet(
        df_test_eval[["union_id","target_binary","predicted","prob_cal","error_type","spatial_stratum"]],
        REPORT_DIR / f"{RUN_ID}_test_predictions_errors.parquet")
    output_paths["test_predictions_errors"] = str(p)

if CFG["write_models"]:
    for mid, res in RESULTS.items():
        joblib.dump(res["model_raw"], MODEL_DIR / f"{RUN_ID}_{mid}_raw.pkl")
        joblib.dump(res["model_cal"], MODEL_DIR / f"{RUN_ID}_{mid}_calibrated.pkl")
    joblib.dump(FEATURE_COLS,        MODEL_DIR / f"{RUN_ID}_feature_cols.pkl")
    joblib.dump(train_medians,       MODEL_DIR / f"{RUN_ID}_train_medians.pkl")
    joblib.dump(clip_bounds,         MODEL_DIR / f"{RUN_ID}_clip_bounds.pkl")
    log(f"  Models + preprocessing artifacts saved to {MODEL_DIR}")

if CFG["write_model_card"]:
    card_lines = [
        f"# Model Card — {PRIMARY_MODEL_ID} | {RUN_ID}",
        "",
        "## Summary",
        f"- Primary model: **{PRIMARY_MODEL_ID}**",
        f"- Calibration method: **{primary['cal_method']}**",
        f"- Threshold policy: **{CFG['threshold_policy']}** (selected on calibration split)",
        f"- Decision threshold: **{primary['threshold']:.6f}**",
        f"- Features retained: **{len(FEATURE_COLS)} / {len(all_feature_cols)}**",
        "",
        "## Held-out Test Metrics",
    ]
    for k, v in primary["metrics_test"].items():
        card_lines.append(f"- {k}: {round(v, 4) if isinstance(v, float) else v}")
    card_lines += [
        "",
        "## Data / Splits",
        f"- Training rows: {len(df_train)}",
        f"- Calibration rows: {len(df_cal)}",
        f"- Test rows: {len(df_test)}",
        f"- Zero-annotation crops (correctly excluded): {len(zero_ann_crops)}",
        "",
        "## Toggle Flags",
        f"- run_optuna: {CFG['run_optuna']}",
        f"- run_lightgbm_family: {CFG['run_lightgbm_family']}",
        f"- run_shap: {CFG['run_shap']}",
    ]
    card_path = CARD_DIR / f"{RUN_ID}_model_card.md"
    card_path.write_text("\n".join(card_lines), encoding="utf-8")
    output_paths["model_card"] = str(card_path)
    log(f"  Model card: {card_path.name}")

manifest = {
    "run_id":                  RUN_ID,
    "notebook_name":           "06_model_training_and_local_resolution.ipynb",
    "created_at_utc":          datetime.now(timezone.utc).isoformat(),
    "primary_model_id":        PRIMARY_MODEL_ID,
    "model_registry_version":  CFG["model_registry_version"],
    "threshold_policy":        CFG["threshold_policy"],
    "feature_count_raw":       len(all_feature_cols),
    "feature_count_core":      len(FEATURE_COLS),
    "zero_annotation_crops":   sorted(zero_ann_crops),
    "toggles": {
        "run_optuna":           CFG["run_optuna"],
        "run_lightgbm_family":  CFG["run_lightgbm_family"],
        "run_shap":             CFG["run_shap"],
        "run_permutation":      CFG["run_permutation_importance"],
        "run_visual_qa":        CFG["run_visual_qa"],
    },
    "split_sizes":  {"train": len(df_train), "cal": len(df_cal), "test": len(df_test)},
    "test_metrics": {mid: res["metrics_test"] for mid, res in RESULTS.items()},
    "thresholds":   {mid: float(res["threshold"]) for mid, res in RESULTS.items()},
    "cal_methods":  {mid: res["cal_method"] for mid, res in RESULTS.items()},
    "inputs":       {"supervision": str(supervision_path), "splits": str(splits_path)},
    "outputs":      output_paths,
}
manifest_path = write_json(manifest, MANIFEST_DIR / f"{RUN_ID}_model_manifest.json")
log(f"  Manifest: {manifest_path.name}")

In [ ]:
# -------------------------------------------------------------------
# EXIT CHECKS
# -------------------------------------------------------------------
log("=" * 90)
log("EXIT CHECKS")

# model_predictions (§3.11)
required_pred_cols = {
    "prediction_id", "union_id", "model_id", "model_version",
    "split_id", "prob_true_spot", "prob_true_spot_calibrated",
    "decision_threshold", "predicted_label", "calibration_method",
}
missing = sorted(required_pred_cols - set(model_predictions.columns))
assert not missing, f"model_predictions missing: {missing}"
assert model_predictions["prediction_id"].is_unique, "prediction_id must be unique"

# No training data in predictions
pred_uids  = set(model_predictions["union_id"])
train_leak = pred_uids & train_uids
assert len(train_leak) == 0, f"Leakage: {len(train_leak)} train union_ids in predictions"

# feature_stats (§3.12)
required_fs = {
    "feature_name", "feature_group", "is_core_frozen",
    "non_null_fraction", "constant_flag", "univariate_score",
    "correlation_pruned_flag", "retained_for_model",
}
missing_fs = sorted(required_fs - set(feature_stats_nb06.columns))
assert not missing_fs, f"feature_stats missing: {missing_fs}"

retained_null = feature_stats_nb06[
    feature_stats_nb06["retained_for_model"] &
    feature_stats_nb06["univariate_score"].isna()
]
assert len(retained_null) == 0, f"{len(retained_null)} retained features have null univariate_score"

log(f"\n{'=' * 90}")
log(f"NOTEBOOK 06 COMPLETE")
log(f"  Primary model:          {PRIMARY_MODEL_ID}")
log(f"  Threshold policy:       {CFG['threshold_policy']} (selected on cal split)")
log(f"  Core frozen features:   {len(FEATURE_COLS)} / {len(all_feature_cols)}")
log(f"  Zero-annotation crops:  {len(zero_ann_crops)} (correctly handled)")
for mid, res in RESULTS.items():
    m = res["metrics_test"]
    log(f"  [{mid:20s}]  AUC={m['roc_auc']:.3f}  AP={m['avg_precision']:.3f}  "
        f"F1={m['f1']:.3f}  Brier={m['brier']:.4f}  thr={res['threshold']:.3f}")
log(f"  model_predictions:      {len(model_predictions):,} rows")
log(f"{'=' * 90}")